# Nigeria Inflation Forecast

Nigerian households and small businesses set prices, salaries and savings without knowing where inflation is heading.
Headline inflation peaked above 30% in 2024, the CPI was rebased in January 2025, and inflation has since fallen to about 14.6% (September 2026).
**Will that decline continue, level off, or reverse?**

In this notebook we pull Nigeria's monthly inflation, crude oil prices and the naira/dollar exchange rate from the Central Bank of Nigeria (CBN),
explore them, compare forecasting models against a naive baseline, and forecast the next 6–12 months with an uncertainty range.

## 1. Setup

We load the libraries we need:

- `pandas`: tables (DataFrames) and time-series tools
- `requests`: downloads data from a web address (URL)
- `matplotlib.pyplot`: draws charts
- `Path`: builds file paths that work on Windows, Mac and Linux

We also fix one colour per series, so headline is always blue, food always orange and core always green in every chart.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

# Chart colours used throughout the notebook (one fixed colour per series)
HEADLINE_COLOUR = "#2a78d6"  # blue
FOOD_COLOUR = "#eb6834"      # orange
CORE_COLOUR = "#1baf7a"      # green
REBASE_COLOUR = "#52514e"    # dark grey, for the January 2025 rebasing marker

## 2. Data pull

**Question:** Can we pull Nigeria's monthly inflation, crude oil price and naira/dollar rate straight from the CBN and combine them into one monthly table?

The data pages on cbn.gov.ng load their numbers from hidden **JSON feeds**. JSON is a text format for data: here, each feed is a long list of
records, and each record (one month or one day) becomes a row in our table.

These feeds are **undocumented**, so the CBN could change or remove them without warning. That's why we always save a CSV copy after a
successful pull, and fall back to that copy if the download fails.

In [3]:
# The three hidden JSON feeds behind cbn.gov.ng data pages (undocumented, so they may change)
BASE_URL = "https://www.cbn.gov.ng/api/"
INFLATION_ENDPOINT = "GetAllInflationRates"
CRUDE_ENDPOINT = "GetAllDailyCrude"
FX_ENDPOINT = "GetAllExchangeRates"

# Some servers refuse requests that don't look like they come from a web browser,
# so we send the same "User-Agent" label a normal browser would
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/130.0 Safari/537.36"
}

# Where we keep our saved copy of the cleaned monthly table
CSV_PATH = Path("data") / "nigeria_macro.csv"

### 2.1 Download the three feeds

`requests.get()` asks the server for a URL and returns a **response**. Every response has a **status code**: `200` means OK,
`403` means "forbidden" (blocked), `404` means "not found". `raise_for_status()` turns any bad status code into an error, so a
failed download can't slip through silently. `timeout=60` means: give up after 60 seconds instead of waiting forever.

We wrap the downloads in `try` / `except`. If **anything** goes wrong (no internet, the server blocks us, the reply isn't valid JSON),
we print the problem and set `live_data = False`. Later cells then load the saved CSV instead of crashing.

In [4]:
def fetch_json(endpoint):
    # Download one CBN feed and return it as a list of records (dictionaries)
    url = BASE_URL + endpoint
    response = requests.get(url, headers=HEADERS, timeout=60)
    response.raise_for_status()  # stop with an error if the server says no (e.g. 403 or 404)
    records = response.json()
    print(f"{endpoint}: {len(records):,} records")
    return records

In [5]:
try:
    inflation_raw = fetch_json(INFLATION_ENDPOINT)
    crude_raw = fetch_json(CRUDE_ENDPOINT)
    fx_raw = fetch_json(FX_ENDPOINT)
    live_data = True
except (requests.RequestException, ValueError) as error:
    # RequestException = network problem or bad status code; ValueError = the reply wasn't valid JSON
    print("Could not get live data from the CBN API:", error)
    print("We'll load the saved CSV copy instead.")
    live_data = False

GetAllInflationRates: 285 records
GetAllDailyCrude: 3,978 records
GetAllExchangeRates: 62,450 records


Let's look at one raw record from each feed before cleaning anything.

*Cells that start with `if live_data:` only do something when the download worked.*

In [6]:
if live_data:
    print(inflation_raw[0])
    print(crude_raw[0])
    print(fx_raw[0])

{'id': 298, 'tyear': 2026, 'tmonth': 9, 'period': 'September 2026', 'allItemsYearOn': '14.63', 'allItemsAverage': '15.80', 'foodYearOn': '19.59', 'foodAverage': '15.72', 'allItemsLessFrmProdYearOn': '12.78', 'allItemsLessFrmProdAverage': '15.91', 'allItemsLessFrmProdAndEnergyYearOn': '12.95', 'allItemsLessFrmProdAndEnergyAvg': '16.54'}
{'id': 4998, 'postDate': '22/10/2026', 'crudeOilPrice': '123.9700'}
{'id': 65222, 'currency': 'CFA', 'ratedate': '2026-10-22', 'buyingrate': '2.2650', 'centralrate': '2.2750', 'sellingrate': '2.2850'}


**Finding:** All three feeds downloaded (285 monthly inflation records, about 4,000 daily crude prices and about 62,000 daily exchange rates).
Three things to fix while cleaning:
1. **Every number is text.** Notice the quotes in `'allItemsYearOn': '14.63'`. We can't do maths on text, so we convert with `pd.to_numeric`.
2. **Newest first.** Time series should run oldest → newest.
3. **Different date styles.** Inflation has separate year and month numbers, crude uses `dd/mm/yyyy`, and exchange rates use `yyyy-mm-dd`.

### 2.2 Clean the inflation feed

We keep three **year-on-year** rates. "Year-on-year" means: how much higher prices are than in the same month last year.

| Feed column | Our name | Meaning |
|---|---|---|
| `allItemsYearOn` | `Headline` | All goods and services |
| `foodYearOn` | `Food` | Food only |
| `allItemsLessFrmProdAndEnergyYearOn` | `Core` | Everything except farm produce and energy, the less volatile part |

We build a proper date from `tyear` and `tmonth`, using **day 1** of each month. Every series in this project will use
month-start dates, so they line up exactly when we combine them.

In [7]:
def clean_inflation(records):
    # Turn the raw inflation records into a monthly table, oldest month first
    df = pd.DataFrame(records)

    # Build a real date from the year and month columns (day 1 = "month start")
    df["Date"] = pd.to_datetime({"year": df["tyear"], "month": df["tmonth"], "day": 1})

    # Keep only the year-on-year rates we need, with short names
    df = df.rename(columns={
        "allItemsYearOn": "Headline",
        "foodYearOn": "Food",
        "allItemsLessFrmProdAndEnergyYearOn": "Core",
    })
    df = df[["Date", "Headline", "Food", "Core"]]

    # Convert text like "14.63" to numbers; anything that isn't a number becomes NaN (blank),
    # which our checks in section 2.6 will catch
    for column in ["Headline", "Food", "Core"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    # The feed is newest-first; time series should run oldest-first
    df = df.set_index("Date").sort_index()
    return df

In [8]:
if live_data:
    inflation = clean_inflation(inflation_raw)
    print(inflation.dtypes)
    display(inflation.tail())

Headline    float64
Food        float64
Core        float64
dtype: object


,Headline,Food,Core
Date,,,
2026-05-01,15.93,16.96,16.82
2026-06-01,15.91,17.52,15.92
2026-07-01,15.43,20.31,14.97
2026-08-01,15.39,19.57,13.29
2026-09-01,14.63,19.59,12.95


**Finding:** The inflation table has 285 months, from January 2003 to September 2026, with no gaps. The three columns are now
`float64` (decimal numbers) instead of text. In September 2026, headline inflation was **14.63%**, food **19.59%** and core **12.95%**.

### 2.3 Clean the crude oil feed

This is the daily price of **Bonny Light**, Nigeria's main export crude, in US dollars per barrel. Two techniques are new here:

- **An explicit date format.** A date like `03/04/2020` could be 3 April or 4 March. We tell pandas the exact format (`%d/%m/%Y`
  = day/month/year) so it can't guess wrong.
- **Resampling.** Inflation is monthly but crude is daily. `resample("MS").mean()` groups all the days in each month and takes the
  average. `"MS"` means "label each group by its **M**onth **S**tart", so January 2020's average gets the date 2020-01-01, the same
  as January 2020's inflation.

In [9]:
def clean_crude(records):
    # Turn daily crude oil prices into monthly average prices (US$ per barrel)
    df = pd.DataFrame(records)

    # Dates look like "22/10/2026" (day/month/year), so give pandas the exact format
    df["Date"] = pd.to_datetime(df["postDate"], format="%d/%m/%Y")
    df["Crude_USD"] = pd.to_numeric(df["crudeOilPrice"], errors="coerce")

    # Average all the days in each month (this also absorbs the few duplicate days in the feed)
    monthly = df.set_index("Date")["Crude_USD"].resample("MS").mean()
    return monthly.round(2)

In [10]:
if live_data:
    crude_monthly = clean_crude(crude_raw)
    print("First month:", crude_monthly.index.min().date(), "| Last month:", crude_monthly.index.max().date())

    # resample() creates a row for every month, so a month with no daily prices shows up as NaN
    empty_months = crude_monthly[crude_monthly.isna()].index.strftime("%Y-%m").tolist()
    print("Months with no crude prices:", empty_months)
    display(crude_monthly["2020-01":"2020-06"])

First month: 2009-10-01 | Last month: 2026-10-01
Months with no crude prices: ['2019-06', '2019-07']


Date
2020-01-01    66.44
2020-02-01    58.46
2020-03-01    33.02
2020-04-01    14.28
2020-05-01    27.62
2020-06-01    40.23
Freq: MS, Name: Crude_USD, dtype: float64

**Finding:** Crude prices start in **October 2009**, so the combined table will have no crude price for 2003 to September 2009.
**June and July 2019 are missing** from the CBN feed entirely. We'll decide how to handle these gaps in Phase 3.
The April 2020 average is very low, with a few days at $7–10 a barrel. That's not an error: it's the COVID-19 oil price
crash, when lockdowns cut global oil demand.

### 2.4 Clean the exchange-rate feed

This feed has daily official rates for **every** currency, and we only want the US dollar. CLAUDE.md warns that the currency names are messy,
so let's **look before we filter**. `repr()` prints each name with quote marks around it, which makes hidden spaces and tabs (`\t`) visible.

In [11]:
if live_data:
    fx = pd.DataFrame(fx_raw)
    # repr() shows hidden characters, e.g. 'US DOLLAR ' with a trailing space
    for name in sorted(fx["currency"].unique()):
        print(repr(name))

'CFA'
'CFA '
'DANISH KRONA'
'DANISH KRONER'
'EURO'
'EURO\t'
'EURO '
'JAPANESE YEN'
'NAIRA'
'POESO'
'POUND STERLING'
'POUNDS STERLING'
'RIYAL'
'RIYAL '
'SDR'
'SDR\t'
'SDR '
'SOUTH AFRICAN RAND'
'SWISS FRANC'
'SWISS FRANC\t'
'UAE DIRHAM'
'US DOLLAR'
'US DOLLAR '
'WAUA'
'WAUA '
'YEN'
'YEN '
'YUAN/RENMINBI'


**Finding:** The names are messy. There's `'US DOLLAR'` *and* `'US DOLLAR '` (trailing space), `'EURO\t'` (a hidden tab),
and spelling variants like `'DANISH KRONA'` / `'DANISH KRONER'` and `'POUND STERLING'` / `'POUNDS STERLING'`.
If we filtered for `"US DOLLAR"` without cleaning, we'd silently lose the rows with a space. `.str.strip()` removes spaces and tabs
from both ends of each name, which is all the US dollar needs.

Each day has three rates: **buying**, **selling**, and **central** (the official midpoint between them). We use the **central rate**,
in naira per US dollar, then average each month just like crude.

In [12]:
def clean_usd_ngn(records):
    # Pick out the US dollar rows and turn daily rates into monthly averages (naira per dollar)
    df = pd.DataFrame(records)

    # Remove hidden spaces/tabs so "US DOLLAR " and "US DOLLAR" count as the same currency
    df["currency"] = df["currency"].str.strip()
    usd = df[df["currency"] == "US DOLLAR"].copy()

    usd["Date"] = pd.to_datetime(usd["ratedate"])  # already in yyyy-mm-dd format
    usd["USD_NGN"] = pd.to_numeric(usd["centralrate"], errors="coerce")

    monthly = usd.set_index("Date")["USD_NGN"].resample("MS").mean()
    return monthly.round(2)

In [13]:
if live_data:
    usd_ngn_monthly = clean_usd_ngn(fx_raw)
    print("First month:", usd_ngn_monthly.index.min().date(), "| Last month:", usd_ngn_monthly.index.max().date())
    print("Months with no rate:", usd_ngn_monthly.isna().sum())
    # Around the June 2023 float of the naira
    display(usd_ngn_monthly["2023-03":"2023-09"])

First month: 2001-12-01 | Last month: 2026-10-01
Months with no rate: 0


Date
2023-03-01    460.43
2023-04-01    460.42
2023-05-01    460.70
2023-06-01    607.75
2023-07-01    769.82
2023-08-01    761.62
2023-09-01    761.33
Freq: MS, Name: USD_NGN, dtype: float64

**Finding:** The US dollar rate runs from December 2001 with no missing months. The June 2023 float is easy to see: about
**₦461** per dollar in May 2023, **₦608** in June and **₦770** in July. This is a jump of roughly two-thirds in two months. Jumps like this
are **policy decisions**, not random noise, and we'll keep that in mind when modelling.

### 2.5 Combine into one monthly table, save it, or fall back to the saved copy

We use the inflation table as the "spine" and **join** the crude and FX columns onto it, matching rows by `Date`.
`join` keeps only the months in the inflation table, so months the other feeds have but inflation doesn't
(e.g. October 2026, which is only partly over) drop out.

If the download worked, we save the result to `data/nigeria_macro.csv`. If it didn't, we load the last saved copy.

In [14]:
if live_data:
    inflation = clean_inflation(inflation_raw)
    crude_monthly = clean_crude(crude_raw)
    usd_ngn_monthly = clean_usd_ngn(fx_raw)

    macro = inflation.join(crude_monthly).join(usd_ngn_monthly)

    CSV_PATH.parent.mkdir(exist_ok=True)  # make sure the data/ folder exists
    macro.to_csv(CSV_PATH)
    print("Live data pulled and saved to", CSV_PATH)
else:
    macro = pd.read_csv(CSV_PATH, index_col="Date", parse_dates=True)
    print("Loaded saved copy from", CSV_PATH)

Live data pulled and saved to data\nigeria_macro.csv


### 2.6 Check the combined table

After every merge we check four things: the **date range**, **missing months**, **duplicate months** and **blank values**.
A merge can silently create any of these problems.

In [15]:
print("Rows, columns:", macro.shape)
print("From", macro.index.min().date(), "to", macro.index.max().date())

# Every month between the first and last date should appear exactly once
all_months = pd.date_range(macro.index.min(), macro.index.max(), freq="MS")
print("Missing months:", len(all_months.difference(macro.index)))
print("Duplicate months:", macro.index.duplicated().sum())

# Count blank (NaN) values in each column
print("\nBlank values per column:")
print(macro.isna().sum())

Rows, columns: (285, 5)
From 2003-01-01 to 2026-09-01
Missing months: 0
Duplicate months: 0

Blank values per column:
Headline      0
Food          0
Core          0
Crude_USD    83
USD_NGN       0
dtype: int64


In [16]:
display(macro.head())
display(macro.tail())

,Headline,Food,Core,Crude_USD,USD_NGN
Date,,,,,
2003-01-01,10.6,7.5,13.6,NaN,126.59
2003-02-01,7.3,3.5,12.1,NaN,127.00
2003-03-01,5.9,0.1,13.9,NaN,126.86
2003-04-01,8.3,3.3,17.6,NaN,126.98
2003-05-01,8.7,3.3,15.5,NaN,127.17


,Headline,Food,Core,Crude_USD,USD_NGN
Date,,,,,
2026-05-01,15.93,16.96,16.82,112.04,1369.49
2026-06-01,15.91,17.52,15.92,87.70,1366.49
2026-07-01,15.43,20.31,14.97,84.81,1372.75
2026-08-01,15.39,19.57,13.29,94.64,1352.47
2026-09-01,14.63,19.59,12.95,117.10,1326.81


**Finding:** We now have one clean monthly table: **285 months** (January 2003 to September 2026) with columns `Headline`, `Food`, `Core`,
`Crude_USD` and `USD_NGN`, saved to `data/nigeria_macro.csv`. There are no missing or duplicate months. The only blanks are in
`Crude_USD`: 81 months before the crude feed starts (January 2003 to September 2009) plus the June and July 2019 gap, so **83** in total.

Still to come in Phase 3 (data quality): add the `Rebased` marker for the January 2025 CPI rebasing, decide what to do with the crude gaps,
and spot-check the CBN numbers against NBS press releases.

## 3. Data quality

**Question:** Can we trust this table enough to build forecasts on it?

A model can't tell a real price shock from a data error, so we check the data *before* modelling:

1. Are the types and value ranges sensible?
2. Are there any copy errors (a month that exactly repeats the one before)?
3. Does the CBN feed agree with the official NBS press releases?
4. How do we fill the gaps?
5. How do we handle the January 2025 CPI rebasing?

### 3.1 Types and value ranges

`dtypes` shows each column's type, and `describe()` gives a quick summary of each column: count, average (`mean`),
spread (`std`), smallest (`min`), largest (`max`) and the quarter points in between.

In [ ]:
print(macro.dtypes)
macro.describe().round(2)

In [ ]:
# Months where food or core inflation was negative (prices lower than a year before)
macro[(macro["Food"] < 0) | (macro["Core"] < 0)]

**Finding:** All five columns are numbers (`float64`). Headline inflation ranges from **3.0%** to **34.8%**, and the naira went from
about ₦114 to over ₦1,600 per dollar. Those are big ranges but real ones. The only surprise is **negative food and core rates** in 2004–2008
(core fell as low as −6.5% in December 2007). The early years of the series (2003–2008) jump around a lot from month to month.
That's how the published series looks, not a mistake in our download, but it means those years are less reliable for modelling.

### 3.2 Copy errors: does any month repeat the month before?

Inflation rates are published to two decimal places, so it would be a big coincidence for headline, food **and** core all to be exactly
the same as the previous month. `diff()` subtracts the previous month from each month. If all three differences are 0, the row is a repeat.

In [ ]:
inflation_columns = ["Headline", "Food", "Core"]
change_from_last_month = macro[inflation_columns].diff()

# A repeat = all three columns unchanged (the first month has nothing to compare with, so skip it)
is_repeat = (change_from_last_month == 0).all(axis=1)
macro.loc["2025-04":"2025-07", inflation_columns].assign(repeat_of_last_month=is_repeat)

**Finding:** Only one month in 23 years is an exact repeat: **June 2025 is identical to May 2025** (26.06 / 24.55 / 24.92).
The NBS press releases for those two months gave different numbers (22.97% in May and 22.22% in June), so June in the CBN feed is almost
certainly a copy error. We'll fix it in 3.4.

### 3.3 Spot-check: CBN feed vs NBS press releases

The **NBS** (National Bureau of Statistics) calculates Nigeria's inflation and announces it in a monthly press release. The CBN feed is
a copy of those numbers, so they should match. CLAUDE.md flagged one month that doesn't: January 2025 is **27.61%** in the feed but
**24.48%** in the NBS announcement. Let's check more months.

Below are the headline rates **as first published** by NBS, taken from news reports of each press release. For two months, a later NBS
release quotes a **revised** figure for the same month, so we record that too.

Sources: [Dec 2024](https://proshare.co/articles/nigerias-inflation-rate-rises-to-34.80-in-december-2024-says-nbs),
[Jan 2025](https://thecable.ng/breaking-nigerias-inflation-rate-drops-from-34-8-to-24-48-after-rebasing-cpi),
[May 2025](https://gazettengr.com/nigerias-inflation-rate-dropped-to-22-97-in-may-nbs/),
[Jun 2025](https://gazettengr.com/nigerias-inflation-rate-drops-to-22-22-in-june-says-nbs/),
[Jul 2025](https://guardian.ng/news/headline-inflation-slows-to-21-88-in-july-2/),
[Aug 2025](https://allafrica.com/stories/202509160075.html),
[Sep 2025](https://nannews.ng/nigerias-inflation-rate-eases-to-18-02-in-september-nbs/),
[Oct and Nov 2025](https://von.gov.ng/?p=503754),
[Dec 2025, revised Nov](https://von.gov.ng/nigerias-inflation-rate-eases-to-15-15-percent-nbs/),
[Sep 2026, revised Sep 2025](https://businessday.ng/uncategorized/article/nigerias-inflation-drops-to-14-63-as-food-prices-fall-in-september/),
[the January 2026 correction](https://stopsandgaps.substack.com/p/the-nbs-backtracks).

In [ ]:
# Headline inflation (%) as published by NBS; None = no revised figure found
nbs_releases = pd.DataFrame({
    "Date": ["2024-12-01", "2025-01-01", "2025-05-01", "2025-06-01", "2025-07-01", "2025-08-01",
             "2025-09-01", "2025-10-01", "2025-11-01", "2025-12-01", "2026-09-01"],
    "NBS_first_release": [34.80, 24.48, 22.97, 22.22, 21.88, 20.12, 18.02, 16.05, 14.45, 15.15, 14.63],
    "NBS_revised": [None, None, None, None, None, None, 20.98, None, 17.33, None, None],
})
nbs_releases["Date"] = pd.to_datetime(nbs_releases["Date"])
spot_check = nbs_releases.set_index("Date")

# Put the CBN feed's number next to each NBS number and measure the gap
spot_check["CBN_feed"] = macro.loc[spot_check.index, "Headline"]
spot_check["CBN_minus_first_release"] = (spot_check["CBN_feed"] - spot_check["NBS_first_release"]).round(2)
spot_check

**Finding:** There's a clear pattern:

- **December 2024 and earlier** (old CPI) match exactly.
- **January to November 2025:** the CBN feed is about **3 percentage points higher** than what NBS first announced
  (food and core too: January 2025 food is 29.63% in the feed vs 26.08% announced, and core is 25.27% vs 22.59%).
- **Where NBS later published a revised figure, it matches the CBN feed exactly** (September 2025: 20.98%, November 2025: 17.33%).
- **December 2025 onwards** match the first releases.

So the gap isn't a CBN mistake. In **January 2026, NBS corrected an error in its newly rebased CPI** and revised its 2025 figures upward,
and the CBN feed holds the **corrected** numbers. We'll keep the CBN series: it's the latest official version, and it's consistent from
month to month. But anyone comparing our charts with 2025 news headlines will see different numbers, so we'll explain this in the README.

The one exception is **June 2025**: the feed copies May's numbers, and we don't have the revised June figure, so we'll estimate it from its neighbours.

### 3.4 Filling the gaps: interpolation

We have three kinds of gaps:

| Gap | Cause | What we do |
|---|---|---|
| June 2025 inflation | Copy error in the feed | Blank it, then fill it in |
| June–July 2019 crude price | Missing from the CBN feed | Fill it in |
| Crude before October 2009 | The feed starts in 2009 | **Leave blank** |

**Interpolation** fills a gap by drawing a straight line between the known values on either side and reading off the missing points.
For one or two missing months in the *middle* of a series, that's a reasonable estimate.

`limit_area="inside"` tells pandas to fill **only** gaps that have real values on both sides. The six years of crude prices before
October 2009 have nothing before them, so filling them would mean inventing data. We leave them blank. Any model that uses crude oil will
simply start in October 2009.

In [ ]:
# June 2025 is an exact copy of May 2025 in the feed: blank it out first...
macro.loc["2025-06-01", inflation_columns] = float("nan")

# ...then fill each gap with a straight line between its neighbours (only gaps with data on both sides)
macro[inflation_columns] = macro[inflation_columns].interpolate(limit_area="inside")
macro["Crude_USD"] = macro["Crude_USD"].interpolate(limit_area="inside")

display(macro.loc["2025-05":"2025-07", inflation_columns])
display(macro.loc["2019-05":"2019-08", ["Crude_USD"]])

**Finding:** June 2025 is now halfway between May and July (headline **25.50%**), instead of a copy of May. The June and July 2019 crude prices
now step down evenly from \$73.34 in May to \$60.79 in August. These are **estimates**, and we'll say so wherever it matters.

### 3.5 The January 2025 rebasing break

**What is rebasing?** The CPI (Consumer Price Index) tracks the cost of a fixed "basket" of goods and services that a typical household buys.
The cost in the **base year** is set to 100. In January 2025, NBS **rebased** the CPI:

- The base year moved from 2009 to **2024**.
- The basket was updated to match what people buy today. Food's share of the basket was cut to about 40%, from roughly half before.

The year-on-year rate for January 2025 compares January 2025 with January 2024, with both measured on the *new* basket. December 2024's
rate (34.80%) was measured on the *old*, food-heavy basket during a food price spike. So the drop from December to January is
**mostly a change in the measuring tape, not prices suddenly calming down**.

We record this with a **dummy variable**, a column that is just 0 or 1. `Rebased` is 0 before January 2025 and 1 from January 2025 onwards.
Later, a model can use it to learn "the level shifted here" instead of mistaking the drop for normal behaviour.

In [ ]:
# 1 from January 2025 onwards (rebased CPI, 2024 = 100), 0 before
macro["Rebased"] = (macro.index >= "2025-01-01").astype(int)
macro.loc["2024-10":"2025-03"]

In [ ]:
# How unusual was the December 2024 -> January 2025 move? The 5 biggest one-month changes since 2003:
headline_change = macro["Headline"].diff()
headline_change.abs().sort_values(ascending=False).head(5).round(2)

Now let's draw it. We zoom in on 2022 onwards and mark the rebasing with a dashed line and light shading. **Every time chart in this
project will carry this marker.**

In [ ]:
recent = macro.loc["2022-01":]
rebase_date = pd.Timestamp("2025-01-01")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(recent.index, recent["Headline"], color=HEADLINE_COLOUR, linewidth=2, label="Headline")
ax.plot(recent.index, recent["Food"], color=FOOD_COLOUR, linewidth=2, label="Food")
ax.plot(recent.index, recent["Core"], color=CORE_COLOUR, linewidth=2, label="Core")

# Rebasing marker: dashed line at January 2025, light shading for the rebased period
ax.axvline(rebase_date, color=REBASE_COLOUR, linestyle="--", linewidth=1)
ax.axvspan(rebase_date, recent.index.max(), color=REBASE_COLOUR, alpha=0.07)
ax.text(rebase_date, 44, "  CPI rebased (2024 = 100)", color=REBASE_COLOUR, va="top")

ax.set_ylim(0, 45)
ax.set_title("Nigeria inflation, 2022 to 2026: the January 2025 rebasing break")
ax.set_ylabel("Year-on-year inflation (%)")
ax.grid(axis="y", color="#e4e3df")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper left", frameon=False)

fig.savefig("images/rebasing_break.png", dpi=150, bbox_inches="tight")
plt.show()

**Finding:** Headline inflation fell **7.19 points in a single month**, from 34.80% (December 2024) to 27.61% (January 2025). Food fell
10.21 points. That's the second-largest one-month move in the whole series, after July 2005, and it happened exactly when the measuring method
changed. After the break, inflation keeps falling steadily through 2025 and levels off at about 15% in 2026, while food inflation turns back up
to about 20%.

**What this means for modelling:** a model trained on 2003–2024 has never seen the new CPI, and a model trained on everything would read the
January 2025 drop as a real shock. As CLAUDE.md asks, we'll try both: **(a) train on 2003–2024 only**, and **(b) train on all data with the
`Rebased` dummy**, and compare them.

### 3.6 Final check

Same checks as in 2.6, on the cleaned table.

In [ ]:
print("Rows, columns:", macro.shape)
print("Duplicate months:", macro.index.duplicated().sum())
print("\nBlank values per column:")
print(macro.isna().sum())

**Finding:** The cleaned table has 285 months and six columns (`Headline`, `Food`, `Core`, `Crude_USD`, `USD_NGN`, `Rebased`), no duplicate months
and no blanks except the **81 months of crude price before October 2009**, which we left blank on purpose. The data is ready for exploring in
Phase 4.

*Note:* `data/nigeria_macro.csv` stays as the raw API snapshot. The cleaning in this section runs again every time the notebook runs, so
each fix is visible and easy to change.